# Deepfake Detector

In [ ]:
!pip install -q kaggle
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
import os

print('Upload your kaggle.json file:')
uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
!unzip -q cifake-real-and-ai-generated-synthetic-images.zip -d /content/cifake
print('Dataset ready!')

In [ ]:
import os
from pathlib import Path

base = Path('/content/cifake')
for split in ['train', 'test']:
    for label in ['REAL', 'FAKE']:
        p = base / split / label
        print(f'{split}/{label}: {len(list(p.glob("*.jpg")))} images')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights
import matplotlib.pyplot as plt
import numpy as np
import time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

IMG_SIZE   = 224
BATCH_SIZE = 64
EPOCHS     = 10
LR         = 1e-4
DATA_DIR   = '/content/cifake'

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/test',  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Classes: {train_dataset.classes}')  # should be [FAKE, REAL]
print(f'Train size: {len(train_dataset)} | Val size: {len(val_dataset)}')

In [ ]:
def build_model():
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    # Freeze all layers first
    for param in model.parameters():
        param.requires_grad = False
    # Replace classifier head for binary classification
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 1)
    )
    # Unfreeze last 3 blocks + classifier
    for name, param in model.named_parameters():
        if any(x in name for x in ['features.7', 'features.8', 'classifier']):
            param.requires_grad = True
    return model

model = build_model().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0.0
    t0 = time.time()
    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_dataset)

    # --- Validate ---
    model.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)
            logits = model(imgs)
            val_loss += criterion(logits, labels).item() * imgs.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
    val_loss /= len(val_dataset)
    val_acc   = correct / len(val_dataset)

    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | {elapsed:.0f}s')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/model.pt')
        print(f'  -> Saved best model (val_acc={best_val_acc:.4f})')

print(f'\nTraining done. Best val accuracy: {best_val_acc:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'],   label='val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history['val_acc'], color='green')
ax2.set_title('Val accuracy'); ax2.set_xlabel('Epoch')
ax2.axhline(best_val_acc, linestyle='--', color='gray', label=f'best={best_val_acc:.3f}')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_curves.png')
plt.show()

In [ ]:
from PIL import Image
import glob, random

# Load best model
model.load_state_dict(torch.load('/content/drive/MyDrive/model.pt', map_location=DEVICE))
model.eval()

# Pick a random test image
fake_imgs = glob.glob('/content/cifake/test/FAKE/*.jpg')
real_imgs = glob.glob('/content/cifake/test/REAL/*.jpg')
samples   = [(p, 'FAKE') for p in random.sample(fake_imgs, 3)] + [(p, 'REAL') for p in random.sample(real_imgs, 3)]
random.shuffle(samples)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (path, true_label) in zip(axes.flat, samples):
    img = Image.open(path).convert('RGB')
    tensor = val_transforms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.sigmoid(model(tensor)).item()
    # class_idx=1 → REAL in ImageFolder (alphabetical: FAKE=0, REAL=1)
    pred_label = 'REAL' if prob > 0.5 else 'FAKE'
    confidence = prob if prob > 0.5 else 1 - prob
    color = 'green' if pred_label == true_label else 'red'
    ax.imshow(img)
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.0%})', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('model.pt is saved to your Google Drive — download it for the backend.')